<a href="https://colab.research.google.com/github/Dhatrisree123/Complaint-Analyzer/blob/main/Complaint_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#link with google genai
!pip install -q google-genai

In [4]:
with open("/content/gemini_key.txt", "r") as file:
    API_KEY = file.read().strip()

if API_KEY:
    print("Gemini API key loaded successfully.")
else:
    print("API key file is empty.")

Gemini API key loaded successfully.


In [5]:
from google import genai

client = genai.Client(api_key=API_KEY)

print("Gemini client initialized successfully.")

Gemini client initialized successfully.


In [7]:
complaint = "My food order arrived two hours late and it was completely cold when delivered."

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=complaint
)

print(response.text)

That is extremely frustrating! Waiting two hours for food only for it to arrive completely cold and inedible is completely unacceptable. 

While I am an AI assistant and can't directly process a refund for you, **you are fully entitled to a full refund** for this experience. Here is the best way to get your money back right now, depending on how you ordered:

### 1. If you used a Delivery App (UberEats, DoorDash, Grubhub, Deliveroo, etc.)
1. **Open the app** and go to your **Orders** history.
2. Select this order and tap **"Help," "Support," or "Report an Issue."**
3. Choose the options for **"Food arrived cold"** and **"Delivery was significantly delayed."**
4. **Demand a full refund, not just a credit.** Automated bots will often try to offer you a $5 or $10 credit. Reject that and ask to speak to a live agent.

**Copy/Paste Script for Chat Support:**
> *"Hi, my order #[Order Number] arrived two hours late and was completely cold and inedible. Because of the severe delay and ruined f

In [20]:
system_instruction = """
You are a Customer Complaint Analyzer.

Your task is to analyze customer complaints strictly based on the information provided.

You must identify:
1. complaint_category
2. severity
3. root_issue
4. recommended_action

GENERAL RULES:
- Use only information explicitly stated or directly supported by the complaint.
- Do not invent facts.
- Do not assume information that is not provided.
- Do not exaggerate the complaint.
- Do not give advice outside the required output fields.
- Keep the root_issue concise and grounded in the complaint.
- The recommended_action must directly address the stated problem.

SEVERITY RULES:

HIGH:
Use High when the complaint describes:
- Significant financial loss or an unresolved financial problem.
- Duplicate or incorrect charges involving money.
- Major service failure.
- A substantial delay that seriously affects the service received.
- Severe damage or a serious safety issue.
- A major problem that significantly prevents the customer from receiving the expected service.

MEDIUM:
Use Medium when:
- The customer experienced a meaningful inconvenience.
- The service was affected but the impact is limited.
- The problem is noticeable but does not represent a major service failure or significant financial loss.

LOW:
Use Low when:
- The complaint describes a minor inconvenience.
- The issue has little practical impact.
- The customer is mainly providing minor feedback or a small request.

IMPORTANT SEVERITY RULE:
Judge severity from the actual impact described in the complaint, not merely from emotional language.

GROUNDING RULE:
If the complaint is unclear, keep the root_issue broad and do not invent missing details.

RECOMMENDED ACTION:
- Recommend an action that directly addresses the complaint.
- Do not promise a refund, replacement, compensation, or other outcome unless the complaint provides a basis for it.
- When appropriate, recommend investigating or verifying the issue before taking action.

Return only the requested analysis.
"""

In [16]:
from google.genai import types

response_schema = {
    "type": "OBJECT",
    "properties": {
        "complaint_category": {
            "type": "STRING",
            "description": "The main category of the customer complaint."
        },
        "severity": {
            "type": "STRING",
            "enum": ["Low", "Medium", "High"],
            "description": "Severity based only on the impact explicitly described."
        },
        "root_issue": {
            "type": "STRING",
            "description": "A concise description of the core problem, grounded strictly in the complaint."
        },
        "recommended_action": {
            "type": "STRING",
            "description": "A practical action appropriate to the stated complaint. Do not invent facts."
        }
    },
    "required": [
        "complaint_category",
        "severity",
        "root_issue",
        "recommended_action"
    ]
}

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=f"""
{system_instruction}

Analyze this customer complaint:

{complaint}
""",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=response_schema
    )
)

print(response.text)

{"complaint_category":"Delivery Issue","severity":"Medium","root_issue":"Food order arrived two hours late and completely cold.","recommended_action":"Issue a refund or credit for the delayed and cold order, and investigate the cause of the delivery delay."}


In [21]:
import json

result = json.loads(response.text)

print("Category:", result["complaint_category"])
print("Severity:", result["severity"])
print("Root Issue:", result["root_issue"])
print("Recommended Action:", result["recommended_action"])

Category: Delivery Issue
Severity: Medium
Root Issue: Food order arrived two hours late and completely cold.
Recommended Action: Issue a refund or credit for the delayed and cold order, and investigate the cause of the delivery delay.


In [18]:
def analyze_complaint(complaint):

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=f"""
{system_instruction}

Analyze this customer complaint:

{complaint}
""",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=response_schema
        )
    )

    return json.loads(response.text)

In [22]:
complaint_1 = """
I was charged twice for the same order, and the second payment has not been refunded.
"""

result_1 = analyze_complaint(complaint_1)

print(json.dumps(result_1, indent=2))

{
  "complaint_category": "Billing",
  "severity": "High",
  "root_issue": "The customer was charged twice for the same order and the duplicate payment has not been refunded.",
  "recommended_action": "Investigate the transaction history to verify the duplicate charge and process the refund for the second payment."
}


In [25]:
test_cases = [
    "I was charged twice for the same order and the second payment has not been refunded.",

    "My electricity was disconnected for three days even though I had already paid my bill.",

    "The delivery arrived 30 minutes late and I had to wait longer than expected.",

    "The website took a little longer than usual to load, but I was still able to place my order.",

    "I am really unhappy with my recent experience."
]

for complaint in test_cases:
    print("\n" + "=" * 60)
    print("Complaint:", complaint)

    try:
        result = analyze_complaint(complaint)
        print(json.dumps(result, indent=2))

    except Exception as e:
        print("API request failed:", e)


Complaint: I was charged twice for the same order and the second payment has not been refunded.
{
  "complaint_category": "Billing",
  "severity": "High",
  "root_issue": "The customer was charged twice for the same order and the duplicate charge has not been refunded.",
  "recommended_action": "Verify the duplicate charge in the billing system and process a refund for the extra payment."
}

Complaint: My electricity was disconnected for three days even though I had already paid my bill.
{
  "complaint_category": "Billing and Service Disconnection",
  "severity": "High",
  "root_issue": "Electricity was disconnected for three days despite prior bill payment.",
  "recommended_action": "Verify payment records, confirm service restoration, and investigate why disconnection occurred after payment was made."
}

Complaint: The delivery arrived 30 minutes late and I had to wait longer than expected.
{
  "complaint_category": "Delivery Delay",
  "severity": "Low",
  "root_issue": "The deliver